In [ ]:

#  Evaluation of Energy-Efficient Architecture Selection on CIFAR-10

!pip -q install thop pandas

import copy
import random
import time
import numpy as np
import pandas as pd
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset, random_split
from thop import profile


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


BATCH_SIZE = 64
GLOBAL_EPOCHS = 5
LOCAL_FINE_TUNE_EPOCHS = 1
LR = 1e-3
NUM_CLASSES = 10
NUM_SERVICES = 50
DIRICHLET_ALPHA = 0.5

# energy model
ENERGY_ALPHA = 2.2e-9      # Joules per FLOP proxy scaling

# ============================================================
# 3. CIFAR-10 Data
# ============================================================
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])

train_full = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform_train
)
test_set = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transform_test
)

train_size = 45000
val_size = 5000
train_set, val_set = random_split(train_full, [train_size, val_size])

global_train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
global_val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
global_test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)



GLOBAL_ARCH = {
    "channels": [64, 64, 64, 64],   # 4 conv blocks
    "kernel_size": 3,
    "num_blocks": 4,
}

def arch_to_string(arch):
    return f"channels={arch['channels']}, k={arch['kernel_size']}, blocks={arch['num_blocks']}"

class AdaptiveCNN(nn.Module):
    def __init__(self, arch, num_classes=10):
        super().__init__()
        channels = arch["channels"]
        k = arch["kernel_size"]
        padding = k // 2

        layers = []
        in_ch = 3
        for i, out_ch in enumerate(channels):
            layers.append(nn.Conv2d(in_ch, out_ch, kernel_size=k, padding=padding, bias=False))
            layers.append(nn.BatchNorm2d(out_ch))
            layers.append(nn.ReLU(inplace=True))
            if i < len(channels) - 1:
                layers.append(nn.MaxPool2d(2))
            in_ch = out_ch

        self.features = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Linear(channels[-1], num_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


@torch.no_grad()
def evaluate_accuracy(model, loader):
    model.eval()
    total = 0
    correct = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        preds = logits.argmax(dim=1)
        total += y.size(0)
        correct += (preds == y).sum().item()
    return 100.0 * correct / max(total, 1)

def train_model(model, train_loader, val_loader, epochs=5, lr=1e-3):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    best_val = 0.0
    best_state = copy.deepcopy(model.state_dict())

    for epoch in range(epochs):
        model.train()
        total = 0
        correct = 0
        running_loss = 0.0

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            preds = logits.argmax(dim=1)
            total += y.size(0)
            correct += (preds == y).sum().item()

        train_acc = 100.0 * correct / max(total, 1)
        val_acc = evaluate_accuracy(model, val_loader)

        if val_acc > best_val:
            best_val = val_acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch [{epoch+1}/{epochs}] | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

    model.load_state_dict(best_state)
    return model

def fine_tune_model(model, loader, epochs=1, lr=1e-3):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    model.train()
    for _ in range(epochs):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
    return model

def compute_flops_params(model):
    dummy = torch.randn(1, 3, 32, 32).to(device)
    flops, params = profile(model, inputs=(dummy,), verbose=False)
    return flops, params

def estimate_energy(flops, local_steps=1, alpha=ENERGY_ALPHA):
    return alpha * flops * local_steps


def extract_labels(dataset):
    labels = []
    for i in range(len(dataset)):
        _, y = dataset[i]
        labels.append(y)
    return np.array(labels)

def dirichlet_partition(dataset, num_clients, alpha=0.5, seed=42):
    rng = np.random.default_rng(seed)
    labels = extract_labels(dataset)
    num_classes = len(np.unique(labels))

    class_indices = [np.where(labels == c)[0] for c in range(num_classes)]
    client_indices = [[] for _ in range(num_clients)]

    for c in range(num_classes):
        idxs = class_indices[c]
        rng.shuffle(idxs)
        proportions = rng.dirichlet(np.repeat(alpha, num_clients))
        split_points = (np.cumsum(proportions) * len(idxs)).astype(int)[:-1]
        splits = np.split(idxs, split_points)
        for client_id, split in enumerate(splits):
            client_indices[client_id].extend(split.tolist())

    return client_indices

train_partitions = dirichlet_partition(train_set, NUM_SERVICES, alpha=DIRICHLET_ALPHA, seed=42)
test_partitions = dirichlet_partition(test_set, NUM_SERVICES, alpha=DIRICHLET_ALPHA, seed=123)


@dataclass
class ServiceState:
    energy_budget: float
    compute_budget_mflops: float
    memory_budget_params: int
    local_steps: int

def make_service_state(idx):
    group = idx % 3
    if group == 0:
        return ServiceState(
            energy_budget=0.95,
            compute_budget_mflops=85.0,
            memory_budget_params=1_000_000,
            local_steps=LOCAL_FINE_TUNE_EPOCHS
        )
    elif group == 1:
        return ServiceState(
            energy_budget=1.10,
            compute_budget_mflops=100.0,
            memory_budget_params=1_300_000,
            local_steps=LOCAL_FINE_TUNE_EPOCHS
        )
    else:
        return ServiceState(
            energy_budget=1.30,
            compute_budget_mflops=120.0,
            memory_budget_params=1_700_000,
            local_steps=LOCAL_FINE_TUNE_EPOCHS
        )


def build_service_loaders():
    service_loaders = []
    for i in range(NUM_SERVICES):
        train_idx = train_partitions[i]
        test_idx = test_partitions[i]

        if len(train_idx) < 20:
            extra = np.random.choice(len(train_set), 50, replace=False).tolist()
            train_idx.extend(extra)
        if len(test_idx) < 10:
            extra = np.random.choice(len(test_set), 20, replace=False).tolist()
            test_idx.extend(extra)

        local_train = Subset(train_set, train_idx)
        local_test = Subset(test_set, test_idx)

        train_loader = DataLoader(local_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
        test_loader = DataLoader(local_test, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

        service_loaders.append((train_loader, test_loader))
    return service_loaders

service_loaders = build_service_loaders()


print("\n==============================")
print("Training Global Architecture")
print("==============================")
global_model = AdaptiveCNN(GLOBAL_ARCH, num_classes=NUM_CLASSES)
global_model = train_model(global_model, global_train_loader, global_val_loader, epochs=GLOBAL_EPOCHS, lr=LR)
global_test_acc = evaluate_accuracy(global_model, global_test_loader)
global_flops, global_params = compute_flops_params(global_model)
global_energy = estimate_energy(global_flops, local_steps=LOCAL_FINE_TUNE_EPOCHS)

print(f"\nGlobal architecture: {arch_to_string(GLOBAL_ARCH)}")
print(f"Global Test Accuracy: {global_test_acc:.2f}%")
print(f"Global FLOPs: {global_flops/1e6:.2f} M")
print(f"Global Params: {global_params/1e6:.4f} M")
print(f"Global Energy: {global_energy:.4f} J")


def clone_arch(arch):
    return {
        "channels": list(arch["channels"]),
        "kernel_size": arch["kernel_size"],
        "num_blocks": arch["num_blocks"],
    }

def feasible(arch, state):
    model = AdaptiveCNN(arch, num_classes=NUM_CLASSES).to(device)
    flops, params = compute_flops_params(model)
    energy = estimate_energy(flops, local_steps=state.local_steps)
    return (
        energy <= state.energy_budget and
        (flops / 1e6) <= state.compute_budget_mflops and
        params <= state.memory_budget_params
    ), flops, params, energy

def prune_one_smallest_channel(arch):
    new_arch = clone_arch(arch)
    idx = int(np.argmax(new_arch["channels"]))
    if new_arch["channels"][idx] > 8:
        new_arch["channels"][idx] = max(8, new_arch["channels"][idx] - 8)
    return new_arch

def prune_by_flops(arch):
    new_arch = clone_arch(arch)
    idx = int(np.argmax(new_arch["channels"]))
    if new_arch["channels"][idx] > 8:
        new_arch["channels"][idx] = max(8, new_arch["channels"][idx] - 16)
    return new_arch

def uniform_scale(arch, scale=0.75):
    new_arch = clone_arch(arch)
    new_arch["channels"] = [max(8, int(c * scale)) for c in new_arch["channels"]]
    return new_arch

def random_prune_until_feasible(arch, state, max_steps=20):
    current = clone_arch(arch)
    for _ in range(max_steps):
        ok, flops, params, energy = feasible(current, state)
        if ok:
            return current, flops, params, energy, True

        idx = random.randint(0, len(current["channels"]) - 1)
        if current["channels"][idx] > 8:
            current["channels"][idx] = max(8, current["channels"][idx] - random.choice([8, 16]))
        else:
            current["channels"][idx] = 8

    ok, flops, params, energy = feasible(current, state)
    return current, flops, params, energy, ok

def flops_based_prune_until_feasible(arch, state, max_steps=20):
    current = clone_arch(arch)
    for _ in range(max_steps):
        ok, flops, params, energy = feasible(current, state)
        if ok:
            return current, flops, params, energy, True
        current = prune_by_flops(current)

    ok, flops, params, energy = feasible(current, state)
    return current, flops, params, energy, ok

def uniform_scale_until_feasible(arch, state, max_steps=20):
    current = clone_arch(arch)
    for _ in range(max_steps):
        ok, flops, params, energy = feasible(current, state)
        if ok:
            return current, flops, params, energy, True
        current = uniform_scale(current, scale=0.85)

    ok, flops, params, energy = feasible(current, state)
    return current, flops, params, energy, ok

def orchnas_progressive_prune_until_feasible(arch, state, beta=1e-3, max_steps=20):
    """
    Energy-aware pruning:
    choose the prune move with best energy-saving / minimal structure damage proxy.
    """
    current = clone_arch(arch)

    for _ in range(max_steps):
        ok, base_flops, base_params, base_energy = feasible(current, state)
        if ok:
            return current, base_flops, base_params, base_energy, True

        candidates = []
        for idx in range(len(current["channels"])):
            cand = clone_arch(current)
            if cand["channels"][idx] > 8:
                cand["channels"][idx] = max(8, cand["channels"][idx] - 8)
                c_ok, c_flops, c_params, c_energy = feasible(cand, state)

                delta_energy = base_energy - c_energy
                delta_complexity = (sum(current["channels"]) - sum(cand["channels"]))
                score = delta_complexity - beta * delta_energy * 1000.0
                candidates.append((score, cand, c_flops, c_params, c_energy, c_ok))

        if not candidates:
            break

        candidates = sorted(candidates, key=lambda x: x[0])
        _, best_arch, best_flops, best_params, best_energy, best_ok = candidates[0]
        current = best_arch

    ok, flops, params, energy = feasible(current, state)
    return current, flops, params, energy, ok


def evaluate_selection_method(method_name, adapt_fn=None):
    accs = []
    energies = []
    flops_list = []
    params_list = []
    feasible_count = 0

    print(f"\n==============================")
    print(f"Evaluating: {method_name}")
    print("==============================")

    for service_id in range(NUM_SERVICES):
        state = make_service_state(service_id)
        train_loader, test_loader = service_loaders[service_id]

        if method_name == "No Adaptation":
            local_arch = clone_arch(GLOBAL_ARCH)
            ok, flops, params, energy = feasible(local_arch, state)

        else:
            local_arch, flops, params, energy, ok = adapt_fn(GLOBAL_ARCH, state)

        if ok:
            feasible_count += 1

        local_model = AdaptiveCNN(local_arch, num_classes=NUM_CLASSES).to(device)

        # transfer overlapping weights from global model
        local_state = local_model.state_dict()
        global_state = global_model.state_dict()
        for k in local_state.keys():
            if k in global_state and local_state[k].shape == global_state[k].shape:
                local_state[k] = global_state[k].clone()
        local_model.load_state_dict(local_state, strict=False)

        # light local fine-tuning
        local_model = fine_tune_model(local_model, train_loader, epochs=LOCAL_FINE_TUNE_EPOCHS, lr=LR)
        local_acc = evaluate_accuracy(local_model, test_loader)

        accs.append(local_acc)
        energies.append(energy)
        flops_list.append(flops / 1e6)
        params_list.append(params / 1e6)

        print(
            f"Service {service_id:02d} | "
            f"Acc={local_acc:.2f}% | "
            f"Energy={energy:.4f} J | "
            f"FLOPs={flops/1e6:.2f} M | "
            f"Params={params/1e6:.4f} M | "
            f"Feasible={ok}"
        )

    mean_acc = float(np.mean(accs))
    mean_energy = float(np.mean(energies))
    mean_flops = float(np.mean(flops_list))
    mean_params = float(np.mean(params_list))
    feasibility_rate = 100.0 * feasible_count / NUM_SERVICES
    energy_reduction = 100.0 * (global_energy - mean_energy) / max(global_energy, 1e-12)

    result = {
        "Method": method_name,
        "Personalised_Test_Accuracy (%)": round(mean_acc, 2),
        "Energy_Reduction (%)": round(energy_reduction, 2),
        "Avg_Energy (J)": round(mean_energy, 4),
        "Avg_FLOPs (M)": round(mean_flops, 2),
        "Avg_Params (M)": round(mean_params, 4),
        "Feasibility_Rate (%)": round(feasibility_rate, 2),
    }
    return result


start_time = time.time()

results = []
results.append(evaluate_selection_method("No Adaptation"))
results.append(evaluate_selection_method("Random Pruning", random_prune_until_feasible))
results.append(evaluate_selection_method("FLOPs-Based Pruning", flops_based_prune_until_feasible))
results.append(evaluate_selection_method("Uniform Scaling", uniform_scale_until_feasible))
results.append(evaluate_selection_method("OrchNAS", orchnas_progressive_prune_until_feasible))

end_time = time.time()


results_df = pd.DataFrame(results)
csv_path = "orchnas_architecture_selection_results.csv"
results_df.to_csv(csv_path, index=False)
